# 06 — Compare models and evaluate next-week surge prediction

**Question:** can price history improve detection over simple baselines under expanding-window evaluation?

Your main experiment is completed. **RUN_MODE = "reuse"** loads the saved model and cutoff, verifies prepared-input hashes, and keeps all main artifacts unchanged. The default requires your original `data/06_models/` and prepared data folders; Git contains notebook outputs, not the trained model/data. Restore these files if they are missing. Never rerun training just to recover a missing model after looking at test results.

For a genuinely new initial study, prepare all experiments, select `RUN_MODE = "train"`, then Restart Kernel and Run All. Training refuses to overwrite an existing model. The main selection rules remain unchanged: compare 2025 folds → fit through December 2025 → choose cutoff on January–March 2026 → evaluate once on April–June 2026.

The fixed 10% price-increase label and classifier score cutoff are separate. Weighted scores are uncalibrated, not verified event probabilities. Supplementary sections below were added **after** inspecting the original test and cannot change the reported main experiment. Original saved notebook results are recorded in `reports/completed_main_experiment.md` and Git history.


## 1. Imports and experiment settings
Only `pricecatcher_config.py` is needed beside the notebooks. The repeated modelling calculations are defined below, with no separate modelling module. Keep the original eight candidates, seed and settings for the completed experiment. `prepare.py` is an optional convenience runner for 01–05, not required when running the notebooks interactively.

New preparations identify the content of the integrated data and lookup files independently of manifest timestamps. Reuse always verifies exact saved inputs first. Legacy manifest differences are recorded as unresolved provenance, not silently declared equivalent.


In [ ]:
import json
import hashlib
from pathlib import Path
from importlib import metadata, reload
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import PrecisionRecallDisplay, ConfusionMatrixDisplay

import pricecatcher_config as cfg
cfg = reload(cfg)

# Existing completed experiment: reuse saved files. Use "train" only for a new initial study.
RUN_MODE = "reuse"
if RUN_MODE not in {"reuse", "train"}:
    raise ValueError("RUN_MODE must be 'reuse' or 'train'.")

SEED = 42
N_JOBS = 2
TREES = 150
OUTPUT_DIR = cfg.PROJECT_ROOT / "06_models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FOLDS = ["fold_1", "fold_2", "fold_3"]
CANDIDATES = {
    "lr_c01": ("Logistic Regression", {"C": 0.1}),
    "lr_c1": ("Logistic Regression", {"C": 1.0}),
    "rf_depth8": ("Random Forest", {"max_depth": 8}),
    "rf_depth16": ("Random Forest", {"max_depth": 16}),
    "xgb_depth3": ("XGBoost", {"max_depth": 3}),
    "xgb_depth5": ("XGBoost", {"max_depth": 5}),
    "lgb_leaves15": ("LightGBM", {"num_leaves": 15}),
    "lgb_leaves31": ("LightGBM", {"num_leaves": 31}),
}
print("Candidates:", len(CANDIDATES), "| Tree iterations:", TREES, "| Outputs:", OUTPUT_DIR)

SUPPLEMENT_DIR = OUTPUT_DIR / "supplementary"
SUPPLEMENT_DIR.mkdir(parents=True, exist_ok=True)
if RUN_MODE == "train" and (OUTPUT_DIR / "best_model.joblib").exists():
    raise ValueError("A completed model already exists. Use reuse mode, or a separate PRICECATCHER_DATA_DIR for a new study.")
if RUN_MODE == "reuse" and not (OUTPUT_DIR / "model_manifest.json").exists():
    raise FileNotFoundError(f"No saved main experiment in {OUTPUT_DIR}. Restore your original 06_models folder. For an initial study only, set RUN_MODE='train'.")


## Small reusable calculations
These functions keep repeated checks and metrics consistent. They do not fit a model until called by the experiment cells below.

In [ ]:
# Shared modelling calculations
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,
                             precision_recall_curve, precision_score, recall_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


def load_experiment(project_root, name, dates):
    """Load train/validation only. Test is deliberately loaded later in the notebook."""
    folder = Path(project_root) / "experiments" / name / "05_modelling"
    if not (folder / "experiment_settings.json").exists():
        raise FileNotFoundError(f"Missing {name} preparation. Run python prepare.py --experiment all first.")
    settings = json.loads((folder / "experiment_settings.json").read_text())
    if tuple(settings[key] for key in ["train_end", "validation_end", "test_end"]) != tuple(dates):
        raise ValueError(f"{name}: preparation dates differ from configuration; rerun 02–05.")
    if settings["preparation"]["experiment"] != name:
        raise ValueError(f"{name}: experiment metadata mismatch.")
    frames = {}
    for split in ["train", "validation"]:
        frames[split] = check_frame(pd.read_parquet(folder / f"{split}.parquet"), settings, split)
    if frames["train"].surge_label.nunique() != 2:
        raise ValueError(f"{name}: training needs both classes. Inspect the coverage/label audits.")
    return folder, settings, frames


def check_frame(frame, settings, split):
    """Validate the modelling contract and time horizon without changing any rows."""
    features = settings["feature_columns"]
    required = features + ["surge_label", "week_start", "next_week_end", "item", "unit"]
    declared = settings["numeric_features"] + settings["categorical_features"]
    if set(declared) != set(features) or len(declared) != len(features):
        raise ValueError("Numeric/categorical columns must partition the declared features exactly.")
    if len(features) != len(set(features)) or any(c.startswith("next_") or c in ["surge_label", "split"] for c in features):
        raise ValueError("Invalid feature list: identifiers/outcomes must not enter X.")
    if frame.empty or set(required) - set(frame.columns):
        raise ValueError(f"{split}: empty data or missing modelling columns.")
    if frame.surge_label.isna().any() or not frame.surge_label.isin([0, 1]).all():
        raise ValueError(f"{split}: labels must be observed 0/1 values.")
    frame = frame.copy()
    for column in ["week_start", "next_week_end"]:
        frame[column] = pd.to_datetime(frame[column])
    if frame[["week_start", "next_week_end"]].isna().any().any():
        raise ValueError("Missing split dates.")
    if not frame.week_start.dt.dayofweek.eq(0).all() or not frame.next_week_end.eq(frame.week_start + pd.Timedelta(days=13)).all():
        raise ValueError("Expected Monday-start weeks and exact next-week outcome horizons.")
    upper = pd.Timestamp(settings[{"train": "train_end", "validation": "validation_end", "test": "test_end"}[split]])
    if not frame.next_week_end.le(upper).all():
        raise ValueError(f"{split}: outcome crosses its cutoff.")
    if split != "train":
        lower = pd.Timestamp(settings["train_end" if split == "validation" else "validation_end"])
        if not (frame.week_start + pd.Timedelta(days=6)).gt(lower).all():
            raise ValueError(f"{split}: prediction overlaps the earlier split.")
    keys = ["item_code", "state", "district", "premise_type", "week_start"]
    if frame.duplicated(keys).any():
        raise ValueError(f"{split}: duplicate local series/week rows.")
    if np.isinf(frame[settings["numeric_features"]].to_numpy(dtype=float)).any():
        raise ValueError("Infinite numeric features; inspect preparation.")
    return frame


def model_inputs(frame, settings):
    """Select the whitelist; normalise categorical types without learning from data."""
    X = frame[settings["feature_columns"]].copy()
    for column in settings["categorical_features"]:
        X[column] = X[column].astype("string").fillna("__missing__").astype(str)
    return X


def build_pipeline(family, parameters, settings, y_train, *, seed=42, n_jobs=2, trees=150):
    """A fresh preprocessor and estimator for every candidate/fold."""
    numeric_steps = [("impute", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True))]
    if family == "Logistic Regression":
        numeric_steps.append(("scale", StandardScaler()))
    preprocessing = ColumnTransformer([
        ("numeric", Pipeline(numeric_steps), settings["numeric_features"]),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=True), settings["categorical_features"]),
    ], sparse_threshold=1.0)
    # No SMOTE: weighting uses this training fold only and preserves observed categories.
    if family == "Logistic Regression":
        model = LogisticRegression(class_weight="balanced", max_iter=2000, random_state=seed, **parameters)
    elif family == "Random Forest":
        model = RandomForestClassifier(n_estimators=trees, min_samples_leaf=10,
            class_weight="balanced_subsample", random_state=seed, n_jobs=n_jobs, **parameters)
    elif family == "XGBoost":
        positives = int(np.asarray(y_train).sum())
        if not 0 < positives < len(y_train):
            raise ValueError("XGBoost weighting requires both training classes.")
        model = XGBClassifier(n_estimators=trees, learning_rate=.05, subsample=.8,
            colsample_bytree=.8, tree_method="hist", eval_metric="logloss",
            scale_pos_weight=(len(y_train) - positives) / positives,
            random_state=seed, n_jobs=n_jobs, **parameters)
    elif family == "LightGBM":
        model = LGBMClassifier(n_estimators=trees, learning_rate=.05, min_child_samples=30,
            colsample_bytree=.8, class_weight="balanced", verbosity=-1,
            random_state=seed, n_jobs=n_jobs, **parameters)
    else:
        raise ValueError(f"Unknown model family: {family}")
    return Pipeline([("preprocess", preprocessing), ("model", model)])


def evaluate(y, scores, predictions):
    """AP is sklearn average precision, not trapezoidal PR-AUC. Undefined AP is NaN."""
    y = np.asarray(y, dtype=int)
    scores = np.asarray(scores, dtype=float)
    predictions = np.asarray(predictions, dtype=int)
    if len(y) == 0 or len(scores) != len(y) or len(predictions) != len(y) or not np.isfinite(scores).all():
        raise ValueError("Metrics need aligned nonempty labels, finite scores and predictions.")
    tn, fp, fn, tp = confusion_matrix(y, predictions, labels=[0, 1]).ravel()
    has_positives = bool(y.sum())
    return {"rows": len(y), "positives": int(y.sum()), "prevalence": float(y.mean()),
            "predicted_surges": int(predictions.sum()),
            "average_precision": float(average_precision_score(y, scores)) if len(np.unique(y)) == 2 else np.nan,
            "precision": float(precision_score(y, predictions, zero_division=0)),
            "recall": float(recall_score(y, predictions, zero_division=0)) if has_positives else np.nan,
            "f1": float(f1_score(y, predictions, zero_division=0)) if has_positives else np.nan,
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}


def choose_cutoff(y_validation, scores):
    """Maximise validation F1; break ties with the highest cutoff (fewer alerts)."""
    if len(np.unique(y_validation)) != 2:
        raise ValueError("Cutoff selection needs both validation classes. Inspect labels; do not use test to choose it.")
    precision, recall, cutoffs = precision_recall_curve(y_validation, scores)
    f1 = np.divide(2 * precision[:-1] * recall[:-1], precision[:-1] + recall[:-1],
                   out=np.zeros(len(cutoffs)), where=(precision[:-1] + recall[:-1]) > 0)
    return float(cutoffs[np.flatnonzero(np.isclose(f1, f1.max(), rtol=0, atol=1e-12))[-1]])


def baseline_scores(frame, surge_threshold):
    """A constant classifier and a fixed current-change rule, with no fitted inputs."""
    current_change = frame["weekly_price_change_pct"].to_numpy(dtype=float)
    if not np.isfinite(current_change).all():
        raise ValueError("The current-change baseline requires the prior-week price eligibility rule.")
    return {
        "Always no surge": (np.zeros(len(frame)), np.zeros(len(frame), dtype=int)),
        "Current-change rule": (current_change, (current_change >= surge_threshold).astype(int)),
    }


def grouped_metrics(frame, scores, predictions, level):
    """Report support and undefined metrics explicitly for each geography/food group."""
    results = []
    for group, positions in frame.groupby(level, sort=True).indices.items():
        group = group if isinstance(group, tuple) else (group,)
        results.append({**dict(zip(level, group)), **evaluate(frame.iloc[positions].surge_label,
                        np.asarray(scores)[positions], np.asarray(predictions)[positions])})
    return pd.DataFrame(results)


def food_group_scores(training, frame):
    """Training-only prevalence; unseen groups fall back to the overall training rate."""
    rates = training.groupby("project_group")["surge_label"].mean()
    return frame["project_group"].map(rates).fillna(training["surge_label"].mean()).to_numpy(dtype=float)


def verify_saved_run(output_dir, project_root):
    """Refuse to reuse a different model or different prepared inputs."""
    manifest = json.loads((output_dir / "model_manifest.json").read_text())
    if not (output_dir / "best_model.joblib").exists():
        raise FileNotFoundError("Saved model is missing. Restore the original 06_models folder; notebook outputs cannot recreate it.")
    with (output_dir / "best_model.joblib").open("rb") as handle:
        if hashlib.file_digest(handle, "sha256").hexdigest() != manifest["model_sha256"]:
            raise ValueError("Saved model changed. Restore the original completed run.")
    for relative_path, expected_hash in manifest["input_sha256"].items():
        path = project_root / "experiments" / relative_path.split("/", 1)[0] / "05_modelling" / relative_path.split("/", 1)[1]
        if not path.exists():
            raise FileNotFoundError(f"Frozen input missing: {relative_path}. Restore the original prepared folder; do not regenerate an exposed completed experiment.")
        with path.open("rb") as handle:
            if hashlib.file_digest(handle, "sha256").hexdigest() != expected_hash:
                raise ValueError(f"Prepared input changed: {relative_path}. Restore the original snapshot; do not retune on the exposed test.")
    selection = json.loads((output_dir / "selection.json").read_text())
    if selection != manifest["selection"]:
        raise ValueError("Selection differs from the saved manifest. Restore the original selection.")
    return manifest, selection


def source_snapshot_status(experiments, mode):
    """Compare content when available; legacy run hashes alone cannot prove changed prices."""
    manifest_hashes = {name: s["preparation"]["source_manifest_sha256"] for name, s in experiments.items()}
    content_hashes = {name: s["preparation"].get("source_snapshot_sha256") for name, s in experiments.items()}
    known_content = {value for value in content_hashes.values() if value}
    if all(content_hashes.values()) and len(known_content) == 1:
        status = "matching_content_snapshot"
    elif len(set(manifest_hashes.values())) == 1 and len(known_content) <= 1:
        status = "matching_legacy_manifest"
    else:
        if mode == "train":
            details = ", ".join(f"{name}={value[:12]}" for name, value in manifest_hashes.items())
            raise ValueError("Source snapshots are not consistently identified: " + details
                             + ". For a new study, integrate once and prepare all folds in a separate data folder.")
        status = "historical_source_consistency_unresolved"
        print("PROVENANCE NOTE: saved model and input hashes were verified against the completed run, "
              "but historical source manifests differ. A timestamp may explain this; identical source data "
              "has not been established. Original results are preserved, not validated as a consistent-source study.")
    return {"status": status, "manifest_hashes": manifest_hashes,
            "content_snapshot_hashes": content_hashes}


## 2. Declare metrics and success criteria before training

**Primary selection metric:** mean **average precision (AP)** across the three expanding validation folds. AP here means `sklearn.metrics.average_precision_score`, a step-weighted precision–recall summary, **not trapezoidal PR-AUC**. Report its fold standard deviation as descriptive variability, not a confidence interval. Ties use lower AP standard deviation, then candidate name.

**Secondary metrics:** precision, recall, F1 and confusion counts. During fold comparison ML cutoffs stay at 0.5; final cutoff optimisation happens only after model selection. Precision with no predicted positives is reported as 0. Recall/F1 with no observed positives and AP without both classes are reported as undefined. Group support must accompany scores.

**Baselines:** always predict no surge; and predict next-week surge when this week's observed price increase is already at least the fixed surge threshold. The latter's AP uses current percentage change as its ranking score. Constant-score AP equals prevalence when both classes exist.

**Measurable criteria:** (1) selected model's mean fold AP exceeds both baselines; (2) final test AP exceeds both test baselines; (3) final test F1 exceeds both baseline F1 scores. Show which criteria are met, even when the model loses. These are relative technical benchmarks, not promised numeric scores or evidence of economic impact. Decide any stakeholder minimum recall/precision before accessing test results.


In [ ]:
# Verify frozen inputs FIRST in reuse mode; this never refits or replaces anything.
if RUN_MODE == "reuse":
    saved_manifest, selection = verify_saved_run(OUTPUT_DIR, cfg.PROJECT_ROOT)
# Read metadata only: no test rows or labels are loaded here.
experiment_settings = {}
for name in FOLDS + ["holdout"]:
    path = cfg.PROJECT_ROOT / "experiments" / name / "05_modelling" / "experiment_settings.json"
    if not path.exists():
        raise FileNotFoundError(f"Missing {name}. Run python prepare.py --experiment all first.")
    experiment_settings[name] = json.loads(path.read_text())

reference = experiment_settings["holdout"]
# Learned exclusions/bounds may differ; the target, source snapshot and shared policies must agree.
variable_keys = {"experiment", "train_end", "validation_end", "test_end", "evaluation_end",
                 "apply_documented_exclusions"}
shared_keys = set(cfg.PREPARATION_SETTINGS) - variable_keys
for name, settings in experiment_settings.items():
    preparation = settings["preparation"]
    if any(preparation[key] != cfg.PREPARATION_SETTINGS[key] for key in shared_keys):
        raise ValueError(f"{name}: shared preparation settings changed. Rerun preparation.")
    for key in ["feature_columns", "numeric_features", "categorical_features", "target_status", "surge_threshold_pct"]:
        if settings[key] != reference[key]:
            raise ValueError(f"{name}: incompatible {key}.")
    if name in FOLDS and preparation["apply_documented_exclusions"]:
        raise ValueError("Early folds must retain retrospective manual exceptions whose review dates are unknown.")

source_provenance = source_snapshot_status(experiment_settings, RUN_MODE)
(SUPPLEMENT_DIR / "source_provenance.json").write_text(json.dumps(source_provenance, indent=2))
print("Source provenance:", source_provenance["status"])
SURGE_THRESHOLD = reference["surge_threshold_pct"]
display(pd.DataFrame([
    {"experiment": name, "train_end": s["train_end"], "validation_end": s["validation_end"],
     "test_end": s["test_end"], "manual_exclusions": s["review_policy"]["documented_training_records_excluded"]}
    for name, s in experiment_settings.items()
]))


## 3. Compare candidates on expanding folds

Each fit gets a **new** median imputer, missingness indicators and categorical encoder trained on that fold alone. Logistic Regression additionally scales numeric features using training statistics. Missing categorical values use a fixed token; unseen categories are ignored by one-hot encoding. Numeric features entirely missing in training are retained by the imputer. Item codes are categories.

Class weighting is derived from training labels only. No evaluation labels influence training weights, preprocessing, tree counts or early stopping. Validation inputs may contain price history observed by their own prediction dates, as intended for weekly prediction. Only one fold is held in memory at a time. The preparation policy intentionally keeps the retrospective manual exceptions in the early folds.


In [ ]:
if RUN_MODE == "train":
    fold_rows = []
    for fold in FOLDS:
        folder, settings, frames = load_experiment(cfg.PROJECT_ROOT, fold, cfg.EXPERIMENTS[fold])
        train, validation = frames["train"], frames["validation"]
        y_train = train.surge_label.astype(int)
        y_validation = validation.surge_label.astype(int)
        if y_validation.nunique() != 2:
            raise ValueError(f"{fold}: AP comparison needs both validation classes; inspect the label audits.")
        X_train = model_inputs(train, settings)
        X_validation = model_inputs(validation, settings)
        print(f"{fold}: {len(train):,} training rows; {len(validation):,} validation rows")

        for name, (scores, predictions) in baseline_scores(validation, SURGE_THRESHOLD).items():
            fold_rows.append({"fold": fold, "candidate": name, "family": "Baseline", "fit_seconds": 0.0,
                              **evaluate(y_validation, scores, predictions)})
        for name, (family, parameters) in CANDIDATES.items():
            pipeline = build_pipeline(family, parameters, settings, y_train,
                                      seed=SEED, n_jobs=N_JOBS, trees=TREES)
            started = perf_counter()
            pipeline.fit(X_train, y_train)
            seconds = perf_counter() - started
            scores = pipeline.predict_proba(X_validation)[:, 1]
            metrics = evaluate(y_validation, scores, scores >= 0.5)
            fold_rows.append({"fold": fold, "candidate": name, "family": family,
                              "fit_seconds": seconds, **metrics})
            print(f"  {name}: AP={metrics['average_precision']:.4f}; F1@0.5={metrics['f1']:.4f}")
            del pipeline
        del frames, train, validation, X_train, X_validation

    fold_metrics = pd.DataFrame(fold_rows)
    fold_metrics.to_csv(OUTPUT_DIR / "fold_metrics.csv", index=False)
    fold_summary = fold_metrics.groupby(["candidate", "family"]).agg(
        mean_ap=("average_precision", "mean"), std_ap=("average_precision", "std"),
        scored_folds=("average_precision", "count"), mean_f1=("f1", "mean"),
        mean_recall=("recall", "mean"), mean_precision=("precision", "mean"),
        total_fit_seconds=("fit_seconds", "sum"),
    ).reset_index().sort_values(["mean_ap", "std_ap", "candidate"], ascending=[False, True, True])
    fold_summary.to_csv(OUTPUT_DIR / "fold_summary.csv", index=False)
    display(fold_summary)
else:
    # Saved artifacts were already verified in the metadata preflight.
    fold_metrics = pd.read_csv(OUTPUT_DIR / "fold_metrics.csv")
    fold_summary = pd.read_csv(OUTPUT_DIR / "fold_summary.csv")
    display(fold_summary)
    print("Reusing completed fold results; no fitting performed.")


## 4. Freeze the candidate, then use the 2026 validation period

Choose the strongest ML candidate by the prespecified fold rule; the benchmark checks can still show that a simple baseline is better. Fit that candidate through December 2025; do not fit any part of it on validation/test rows. Choose the **highest score cutoff maximising validation F1** (conservative tie-break). Validation F1 after cutoff tuning is a selection result, not an unbiased performance estimate.

The fitted pipeline is kept unchanged for final test evaluation and later predictions. We do not refit on train+validation, because that would change the score scale after choosing the cutoff. A future deployment refit needs its own later calibration/evaluation protocol.


In [ ]:
if RUN_MODE == "train":
    ranked = fold_summary.loc[fold_summary.candidate.isin(CANDIDATES)].copy()
    if not ranked.scored_folds.eq(len(FOLDS)).all():
        raise ValueError("Every candidate must have AP from all folds; do not select from inconsistent subsets.")
    selected_name = ranked.iloc[0].candidate
    selected_family, selected_parameters = CANDIDATES[selected_name]
    folder, settings, frames = load_experiment(cfg.PROJECT_ROOT, "holdout", cfg.EXPERIMENTS["holdout"])
    train, validation = frames["train"], frames["validation"]
    final_pipeline = build_pipeline(selected_family, selected_parameters, settings, train.surge_label.astype(int),
                                    seed=SEED, n_jobs=N_JOBS, trees=TREES)
    final_pipeline.fit(model_inputs(train, settings), train.surge_label.astype(int))
    validation_scores = final_pipeline.predict_proba(model_inputs(validation, settings))[:, 1]
    score_cutoff = choose_cutoff(validation.surge_label.astype(int), validation_scores)
    print("Selected:", selected_name, "| Validation-selected score cutoff:", score_cutoff)

    validation_rows = [
        {"model": selected_name, "cutoff": score_cutoff,
         **evaluate(validation.surge_label, validation_scores, validation_scores >= score_cutoff)},
        {"model": selected_name + " @0.5", "cutoff": 0.5,
         **evaluate(validation.surge_label, validation_scores, validation_scores >= 0.5)},
    ]
    for name, (scores, predictions) in baseline_scores(validation, SURGE_THRESHOLD).items():
        validation_rows.append({"model": name, **evaluate(validation.surge_label, scores, predictions)})
    validation_metrics = pd.DataFrame(validation_rows)
    validation_metrics.to_csv(OUTPUT_DIR / "validation_metrics.csv", index=False)
    display(validation_metrics)

    # Persist the chosen model and decision BEFORE accessing test rows.
    joblib.dump(final_pipeline, OUTPUT_DIR / "best_model.joblib")
    selection = {"candidate": selected_name, "family": selected_family, "parameters": selected_parameters,
                 "score_cutoff": score_cutoff, "selection_metric": "mean expanding-fold average precision",
                 "cutoff_rule": "maximum validation F1; highest cutoff breaks ties",
                 "seed": SEED, "n_jobs": N_JOBS, "tree_iterations": TREES,
                 "feature_columns": settings["feature_columns"], "target": settings["target"],
                 "surge_threshold_pct": SURGE_THRESHOLD, "target_status": settings["target_status"],
                 "score_interpretation": "uncalibrated model score; not verified event probability",
                 "training_end": settings["train_end"], "validation_end": settings["validation_end"],
                 "test_end": settings["test_end"], "refit_on_validation": False}
    (OUTPUT_DIR / "selection.json").write_text(json.dumps(selection, indent=2))
else:
    # Hashes were checked before loading this trusted local model.
    selected_name = selection["candidate"]
    selected_family = selection["family"]
    selected_parameters = selection["parameters"]
    score_cutoff = selection["score_cutoff"]
    folder, settings, frames = load_experiment(cfg.PROJECT_ROOT, "holdout", cfg.EXPERIMENTS["holdout"])
    train, validation = frames["train"], frames["validation"]
    final_pipeline = joblib.load(OUTPUT_DIR / "best_model.joblib")
    validation_scores = final_pipeline.predict_proba(model_inputs(validation, settings))[:, 1]
    ranked = fold_summary.loc[fold_summary.candidate.eq(selected_name)].copy()
    validation_metrics = pd.read_csv(OUTPUT_DIR / "validation_metrics.csv")
    print("Reusing:", selected_name, "| Frozen score cutoff:", score_cutoff)
    display(validation_metrics)


## 5. Final test — evaluate the frozen choice once

**Only now load April–June 2026 test rows.** No model, feature, threshold or cleaning change follows from these results. If results are weak, report that finding; further development needs a fresh untouched evaluation period. Repeatedly tuning after inspecting this section invalidates the final-test claim.

We report the selected cutoff, the fixed 0.5 reference and both baselines. AP assesses ranking independently of the classification cutoff. Scores from weighted models are not calibrated probabilities.


In [ ]:
test = check_frame(pd.read_parquet(folder / "test.parquet"), settings, "test")
test_scores = final_pipeline.predict_proba(model_inputs(test, settings))[:, 1]
test_predictions = (test_scores >= score_cutoff).astype(int)
test_rows = [
    {"model": selected_name, "cutoff": score_cutoff, **evaluate(test.surge_label, test_scores, test_predictions)},
    {"model": selected_name + " @0.5", "cutoff": 0.5, **evaluate(test.surge_label, test_scores, test_scores >= 0.5)},
]
for name, (scores, predictions) in baseline_scores(test, SURGE_THRESHOLD).items():
    test_rows.append({"model": name, **evaluate(test.surge_label, scores, predictions)})
test_metrics = pd.DataFrame(test_rows)
if RUN_MODE == "train":
    test_metrics.to_csv(OUTPUT_DIR / "test_metrics.csv", index=False)
display(test_metrics)

selected_mean_ap = float(ranked.iloc[0].mean_ap)
baseline_mean_ap = float(fold_summary.loc[fold_summary.family.eq("Baseline"), "mean_ap"].max())
baseline_test = test_metrics.loc[test_metrics.model.isin(["Always no surge", "Current-change rule"])]
criteria = pd.DataFrame([
    {"criterion": "Mean fold AP exceeds both baselines", "model_value": selected_mean_ap,
     "reference_value": baseline_mean_ap},
    {"criterion": "Test AP exceeds both baselines", "model_value": test_metrics.iloc[0].average_precision,
     "reference_value": baseline_test.average_precision.max()},
    {"criterion": "Test F1 exceeds both baselines", "model_value": test_metrics.iloc[0].f1,
     "reference_value": baseline_test.f1.max()},
])
criteria["assessable"] = criteria[["model_value", "reference_value"]].notna().all(axis=1)
criteria["met"] = pd.Series(pd.NA, index=criteria.index, dtype="boolean")
criteria.loc[criteria.assessable, "met"] = (
    criteria.loc[criteria.assessable, "model_value"] > criteria.loc[criteria.assessable, "reference_value"])
if RUN_MODE == "train":
    criteria.to_csv(OUTPUT_DIR / "benchmark_checks.csv", index=False)
display(criteria)


## 6. Check food groups, geography and errors

Overall performance can hide weak groups or be dominated by vegetables. Export support, prevalence and confusion counts alongside metrics for each food group, state and district. Undefined group metrics stay missing; zero observed surges cannot establish positive-case detection quality. These are descriptive test breakdowns, not a new tuning step or proof of geographical fairness.


In [ ]:
group_rows = []
for model_name, scores, predictions in [(selected_name, test_scores, test_predictions)] + [
    (name, scores, predictions) for name, (scores, predictions) in baseline_scores(test, SURGE_THRESHOLD).items()
]:
    for level in [["project_group"], ["state"], ["state", "district"]]:
        table = grouped_metrics(test, scores, predictions, level)
        table.insert(0, "model", model_name)
        table.insert(1, "breakdown", "/".join(level))
        group_rows.append(table)
group_results = pd.concat(group_rows, ignore_index=True)
if RUN_MODE == "train":
    group_results.to_csv(OUTPUT_DIR / "test_group_metrics.csv", index=False)
display(group_results.loc[group_results.breakdown.eq("project_group")])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_predictions(test.surge_label, test_predictions, labels=[0, 1],
    display_labels=["No surge", "Surge"], ax=axes[0], colorbar=False)
axes[0].set_title("Frozen model and validation-selected cutoff")
if test.surge_label.nunique() == 2:
    PrecisionRecallDisplay.from_predictions(test.surge_label, test_scores, name=selected_name, ax=axes[1])
    current_scores, _ = baseline_scores(test, SURGE_THRESHOLD)["Current-change rule"]
    PrecisionRecallDisplay.from_predictions(test.surge_label, current_scores, name="Current-change rule", ax=axes[1])
    axes[1].axhline(test.surge_label.mean(), linestyle="--", color="grey", label="Prevalence reference")
    axes[1].legend()
else:
    axes[1].text(.1, .5, "PR comparison undefined: test has one class")
axes[1].set_title("Final-test precision–recall")
fig.tight_layout()
if RUN_MODE == "train":
    fig.savefig(OUTPUT_DIR / "test_evaluation.png", dpi=160, bbox_inches="tight")
plt.show()


## 7. Save auditable predictions and model provenance

Keep identifiers, dates, labels and scores so false alarms/misses can be inspected without retraining. Latest predictions use the same fitted pipeline and cutoff. These historical-snapshot scores do not imply a live dashboard.

The manifest records prepared-input, code and saved-model hashes alongside package versions. Later SHAP work should load **this saved pipeline** and its training-derived preprocessing, then precompute selected explanations for responsive dashboard display. Isolation Forest remains a separate task. Load only trusted model files; joblib stores executable Python objects.


In [ ]:
if RUN_MODE == "train":
    identifiers = ["item_code", "item", "unit", "project_group", "state", "district", "premise_type", "week_start", "next_week_end"]
    for name, frame, scores in [("validation", validation, validation_scores), ("test", test, test_scores)]:
        predictions = frame[identifiers + ["surge_label"]].copy()
        predictions["model_score"] = scores
        predictions["predicted_surge"] = (scores >= score_cutoff).astype(int)
        predictions.to_parquet(OUTPUT_DIR / f"{name}_predictions.parquet", index=False)

    latest = pd.read_parquet(folder / "latest_prediction_inputs.parquet")
    latest_predictions = latest[identifiers].copy()
    if len(latest):
        latest_predictions["model_score"] = final_pipeline.predict_proba(model_inputs(latest, settings))[:, 1]
        latest_predictions["predicted_surge"] = (latest_predictions.model_score >= score_cutoff).astype(int)
    else:
        latest_predictions["model_score"] = pd.Series(dtype=float)
        latest_predictions["predicted_surge"] = pd.Series(dtype=int)
        print("No supported latest-week rows. No stale local predictions substituted.")
    latest_predictions.to_parquet(OUTPUT_DIR / "latest_predictions.parquet", index=False)

    input_hashes = {}
    for name in FOLDS + ["holdout"]:
        input_folder = cfg.PROJECT_ROOT / "experiments" / name / "05_modelling"
        files = ["experiment_settings.json", "train.parquet", "validation.parquet"]
        if name == "holdout":
            files += ["test.parquet", "latest_prediction_inputs.parquet"]
        for filename in files:
            with (input_folder / filename).open("rb") as handle:
                input_hashes[f"{name}/{filename}"] = hashlib.file_digest(handle, "sha256").hexdigest()
    versions = {}
    for package in ["pandas", "numpy", "scikit-learn", "xgboost", "xgboost-cpu", "lightgbm", "joblib"]:
        try:
            versions[package] = metadata.version(package)
        except metadata.PackageNotFoundError:
            pass  # Only one of xgboost / xgboost-cpu needs to be installed.
    code_hashes = {name: hashlib.sha256((Path(cfg.__file__).parent / name).read_bytes()).hexdigest()
                   for name in ["06_model_training_and_evaluation.ipynb", "05_features_and_labels.ipynb", "pricecatcher_config.py"]}
    with (OUTPUT_DIR / "best_model.joblib").open("rb") as handle:
        model_hash = hashlib.file_digest(handle, "sha256").hexdigest()
    manifest = {"source_provenance": source_provenance, "code_sha256": code_hashes, "model_sha256": model_hash, "selection": selection, "candidates": CANDIDATES, "folds": FOLDS,
                "input_sha256": input_hashes, "package_versions": versions,
                "holdout_preparation": settings["preparation"], "holdout_cleaning": settings["review_policy"],
                "metric_definition": "sklearn average precision; not trapezoidal PR-AUC",
                "limitations": ["Uncalibrated scores", "Observed-median labels can reflect monitoring composition",
                                "Per-group scarcity", "No demonstrated affordability or adoption outcomes"]}
    (OUTPUT_DIR / "model_manifest.json").write_text(json.dumps(manifest, indent=2))
    print("Saved model, results and prediction files to:", OUTPUT_DIR)
else:
    print("Original model, cutoff, predictions and main result files preserved.")


## 8. Supplementary analysis — food-group prevalence baseline
**Added after the original test was exposed.** This is not a fourth prespecified benchmark and never reselects the model, features or main cutoff.

Give each row its food group's surge frequency learned from **training only**. An unseen group uses the overall training frequency. Choose this baseline's own classification cutoff on validation only, then compare ranking and F1 on the same test rows. The baseline uses no recent price movement, so it checks whether the model adds information beyond group composition. Do not change its specification after inspecting these supplementary test results.


In [ ]:
group_validation_scores = food_group_scores(train, validation)
group_test_scores = food_group_scores(train, test)
group_cutoff = choose_cutoff(validation.surge_label.astype(int), group_validation_scores)
supplementary_rows = []
for split, frame, model_scores, prevalence_scores in [
    ("validation", validation, validation_scores, group_validation_scores),
    ("test", test, test_scores, group_test_scores),
]:
    for name, scores, cutoff in [(selected_name, model_scores, score_cutoff),
                                 ("Training food-group prevalence", prevalence_scores, group_cutoff)]:
        supplementary_rows.append({"analysis": "post-hoc supplementary", "split": split,
            "model": name, "score_cutoff": cutoff, **evaluate(frame.surge_label, scores, scores >= cutoff)})
prevalence_comparison = pd.DataFrame(supplementary_rows)
prevalence_comparison.to_csv(SUPPLEMENT_DIR / "food_group_baseline.csv", index=False)
display(prevalence_comparison)


## 9. Supplementary analysis — validation source-quality audit
This is a **diagnostic, not a model-robustness experiment**. Keep the model and cutoff frozen. Join suspicious-record counts for the current and target weeks to validation predictions only. The target-week flags are retrospective audit evidence and never model features. Compare errors for supported pairs with/without flags and report group composition.

Flags are not confirmed source errors. Different error rates do not prove that suspicious records caused model failures. This check does not remove prices, relabel rows, retrain or use test errors to choose a cleaning policy. A future cleaning sensitivity experiment needs a separate design and untouched evaluation period.


In [ ]:
weekly_metadata_path = folder.parent / "04_weekly" / "preparation_settings.json"
if json.loads(weekly_metadata_path.read_text()) != settings["preparation"]:
    raise ValueError("Weekly audit settings differ from the completed main experiment.")
holdout_weekly = pd.read_parquet(folder.parent / "04_weekly" / "weekly_prices.parquet",
    columns=cfg.GROUP_KEYS + ["week_start", "suspicious_record_count", "weekly_median_price"])
holdout_weekly["week_start"] = pd.to_datetime(holdout_weekly["week_start"])
holdout_weekly = holdout_weekly.loc[
    (holdout_weekly.week_start + pd.Timedelta(days=6)) <= pd.Timestamp(settings["validation_end"])].copy()
quality = validation.copy()
quality["model_score"] = validation_scores
quality["predicted_surge"] = (validation_scores >= score_cutoff).astype(int)
quality = quality.merge(holdout_weekly.rename(columns={"suspicious_record_count": "current_flagged_records", "weekly_median_price": "current_source_median"}),
    on=cfg.GROUP_KEYS + ["week_start"], how="left", validate="one_to_one")
quality["next_week_start"] = quality.week_start + pd.Timedelta(days=7)
quality = quality.merge(holdout_weekly.rename(columns={"week_start": "next_week_start",
    "suspicious_record_count": "target_flagged_records", "weekly_median_price": "target_source_median"}),
    on=cfg.GROUP_KEYS + ["next_week_start"], how="left", validate="one_to_one")
if quality[["current_flagged_records", "target_flagged_records"]].isna().any().any():
    raise ValueError("Weekly audit does not match validation pairs. Restore the matching preparation snapshot.")
if not np.isclose(quality.weekly_median_price, quality.current_source_median).all():
    raise ValueError("Weekly audit prices differ from model inputs. Restore the matching snapshot.")
audited_labels = ((quality.target_source_median / quality.current_source_median - 1) * 100 >= SURGE_THRESHOLD).astype(int)
if not audited_labels.eq(quality.surge_label).all():
    raise ValueError("Weekly audit outcomes differ from the completed labels.")
quality["source_quality"] = np.where(
    (quality.current_flagged_records > 0) | (quality.target_flagged_records > 0),
    "either_week_flagged", "neither_week_flagged")
quality_rows = []
for flag, part in quality.groupby("source_quality"):
    quality_rows.append({"source_quality": flag, **evaluate(part.surge_label, part.model_score, part.predicted_surge)})
quality_summary = pd.DataFrame(quality_rows)
quality_summary.to_csv(SUPPLEMENT_DIR / "validation_source_quality.csv", index=False)
quality.groupby(["source_quality", "project_group"]).size().rename("rows").to_csv(
    SUPPLEMENT_DIR / "validation_source_quality_groups.csv")
display(quality_summary)


## 10. Interpret false alerts, missed surges and temporal variability
A useful monitoring prototype can still produce many false alerts. Show false alerts **among alerts** (false discovery proportion) separately from false positives among actual negatives (false positive rate). Pair each period's AP with prevalence/support and its AP-to-prevalence ratio; that ratio is descriptive, not significance evidence. Food groups with tiny support cannot justify stable detection claims. Close candidate AP scores do not establish decisive superiority.


In [ ]:
error_rows = []
for split, frame, scores in [("validation", validation, validation_scores), ("test", test, test_scores)]:
    result = evaluate(frame.surge_label, scores, scores >= score_cutoff)
    alerts = result["tp"] + result["fp"]
    negatives = result["tn"] + result["fp"]
    error_rows.append({"split": split, "true_alerts": result["tp"], "false_alerts": result["fp"],
        "missed_surges": result["fn"], "prevalence": result["prevalence"],
        "false_alerts_among_alerts": result["fp"] / alerts if alerts else np.nan,
        "false_positive_rate": result["fp"] / negatives if negatives else np.nan,
        "missed_surge_fraction": result["fn"] / result["positives"] if result["positives"] else np.nan})
error_summary = pd.DataFrame(error_rows)
error_summary.to_csv(SUPPLEMENT_DIR / "error_interpretation.csv", index=False)
display(error_summary)
temporal = fold_metrics.loc[fold_metrics.candidate.eq(selected_name),
    ["fold", "rows", "positives", "prevalence", "average_precision"]].rename(columns={"fold": "period"})
for period, frame, scores in [("validation_2026", validation, validation_scores), ("test_2026", test, test_scores)]:
    metrics = evaluate(frame.surge_label, scores, scores >= score_cutoff)
    temporal = pd.concat([temporal, pd.DataFrame([{"period": period,
        **{key: metrics[key] for key in ["rows", "positives", "prevalence", "average_precision"]}}])], ignore_index=True)
temporal["ap_to_prevalence"] = temporal.average_precision / temporal.prevalence.replace(0, np.nan)
temporal.to_csv(SUPPLEMENT_DIR / "temporal_context.csv", index=False)
display(temporal)
# Keep provenance for post-hoc outputs separate from the original main manifest.
with (OUTPUT_DIR / "best_model.joblib").open("rb") as handle:
    supplementary_model_hash = hashlib.file_digest(handle, "sha256").hexdigest()
with (folder.parent / "04_weekly" / "weekly_prices.parquet").open("rb") as handle:
    weekly_audit_hash = hashlib.file_digest(handle, "sha256").hexdigest()
(SUPPLEMENT_DIR / "analysis_settings.json").write_text(json.dumps({
    "status": "supplementary after original test exposure", "source_provenance": source_provenance, "main_model_sha256": supplementary_model_hash,
    "main_candidate": selected_name, "main_cutoff": score_cutoff,
    "food_group_baseline_cutoff": group_cutoff,
    "source_quality_scope": "validation only; diagnostic, not retraining sensitivity",
    "main_model_changed": False, "weekly_audit_sha256": weekly_audit_hash,
}, indent=2))


## What to write in the report

Report the fold dates, candidate settings, AP definition, preprocessing/weighting policy, selected candidate and validation cutoff. Present fold variability, final-test baseline comparisons, confusion counts and food-group results. State which prespecified criteria were met; do not call the integration of existing algorithms inherently novel or universally best.

The fold results support candidate selection; validation results after cutoff tuning are not final generalisation evidence. The test is the final estimate for the configured period, subject to temporal dependence and source uncertainty. No affordability, SDG or behaviour-change effect was measured. Future notebooks can implement separate anomaly detection, SHAP precomputation and the prototype dashboard without using test outcomes to retune this experiment.
